# 07 · Guardrails

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Needs:** Ollama.

Instructions ask a model to behave. **Guardrails check** whether it did, or
whether it's about to be asked to do something it shouldn't, and can **stop the
run**. The SDK has guardrails at three points: on the user's **input**, on the
agent's final **output**, and around individual **tool calls**.

**You will learn**
- Input and output guardrails, and the **tripwire** that stops a run
- Guardrails that use a small, fast "checker" agent
- The critical detail: **parallel vs blocking** input guardrails, and a file that got deleted anyway
- **Tool guardrails**: validating arguments before a tool runs, and outputs after
- Layering guardrails, and what they can't do

**Prerequisites:** notebooks 02–04.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

import asyncio, json, re
from pydantic import BaseModel
from agents import (Agent, Runner, function_tool, RunContextWrapper,
                    input_guardrail, output_guardrail, GuardrailFunctionOutput,
                    InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered,
                    tool_input_guardrail, tool_output_guardrail, ToolGuardrailFunctionOutput)
from common import get_model

model = get_model()

## 1 · Why guardrails

A system prompt saying "never discuss X" or "never email outsiders" is a
request to a probabilistic model. Clever input, confusion or plain bad luck can
break it. A guardrail is **code that runs around the agent** and enforces the
rule regardless of what the model decides:

```
 user input ──► [input guardrails] ──► agent loop ──────────────────────────► [output guardrails] ──► reply
                   │ tripwire                 │ each tool call                     │ tripwire
                   ▼                          ▼                                    ▼
          InputGuardrailTripwireTriggered   [tool input guardrail] → tool →   OutputGuardrailTripwireTriggered
                                            [tool output guardrail]
```

A guardrail function returns `GuardrailFunctionOutput(output_info=..., tripwire_triggered=bool)`.
If the tripwire is triggered, the run stops with an exception that your code
catches and turns into a polite refusal, a human escalation, or a log entry.

## 2 · An input guardrail

The cheapest guardrails are plain code: a regex, a length limit, a blocklist.
This one stops users from pasting what looks like a credit-card number:

In [ ]:
CARD = re.compile(r'\b(?:\d[ -]?){13,16}\b')

@input_guardrail
async def no_card_numbers(ctx: RunContextWrapper, agent: Agent, user_input) -> GuardrailFunctionOutput:
    text = user_input if isinstance(user_input, str) else json.dumps(user_input)
    found = bool(CARD.search(text))
    return GuardrailFunctionOutput(output_info={'card_number_found': found}, tripwire_triggered=found)


support = Agent(name='Support', model=model, input_guardrails=[no_card_numbers],
                instructions='You help customers of Nimbus Robotics. Be brief.')

for message in ['When will my robot kit ship?', 'Please charge my card 4111 1111 1111 1111 for the kit.']:
    try:
        result = await Runner.run(support, message)
        print('✅', result.final_output[:100])
    except InputGuardrailTripwireTriggered as err:
        print('⛔ blocked:', err.guardrail_result.output.output_info,
              '→ reply: "For your safety, never share card numbers in chat."')

The second message never reached the model. The card number wasn't sent to a
model provider, logged in its history, or echoed back.

## 3 · A guardrail with its own agent

Some rules need judgement: "is this request in scope?", "is this abusive?". A
guardrail can call a **small, fast agent** with structured output and trip on
its verdict. Cheap model for the check, expensive model for the real work:

In [ ]:
class ScopeCheck(BaseModel):
    in_scope: bool
    reason: str

scope_checker = Agent(name='Scope checker', model=model, output_type=ScopeCheck, instructions=(
    'The assistant only helps with Nimbus Robotics products, orders and deliveries. '
    'Is the user message in scope?'))


@input_guardrail(run_in_parallel=False)          # explained in §4
async def on_topic(ctx: RunContextWrapper, agent: Agent, user_input) -> GuardrailFunctionOutput:
    verdict = (await Runner.run(scope_checker, user_input, context=ctx.context)).final_output
    return GuardrailFunctionOutput(output_info=verdict, tripwire_triggered=not verdict.in_scope)


scoped = Agent(name='Support', model=model, input_guardrails=[on_topic],
               instructions='You help customers of Nimbus Robotics. Be brief.')

for message in ['Can the C2 robot climb curbs?', 'Write my history essay about Napoleon.']:
    try:
        print('✅', (await Runner.run(scoped, message)).final_output[:100])
    except InputGuardrailTripwireTriggered as err:
        print('⛔ out of scope:', err.guardrail_result.output.output_info.reason[:120])

Look closely at the answer that *passed*. In our run the agent confidently
stated a curb-climbing height for the C2, which it invented, since it has no
product data. The guardrail did its job (the question was in scope), and it
says nothing about whether the answer is **true**. Scope and accuracy are
different problems: accuracy comes from grounding (tools, documents, notebook 11).

## 4 · Parallel vs blocking: the detail that matters most

By default an input guardrail runs **in parallel** with the agent
(`run_in_parallel=True`). That's faster: the agent doesn't wait for the check.
But think about what "in parallel" means: **the agent starts working while the
check is still running.** If the agent calls a tool with side effects before the
guardrail finishes, the tool has already run by the time the tripwire fires.

Let's test it. A slow guardrail (4 seconds, like a real LLM check) is meant to
block any request to delete files. The agent has a `delete_file` tool:

In [ ]:
deleted = []                                     # what REALLY happened

@function_tool
def delete_file(path: str) -> str:
    """Delete a file."""
    deleted.append(path)
    return f'deleted {path}'


def ops_agent(parallel: bool) -> Agent:
    @input_guardrail(run_in_parallel=parallel)
    async def block_deletes(ctx, agent, user_input):
        await asyncio.sleep(4)                   # a slow check, e.g. its own model call
        return GuardrailFunctionOutput(output_info=None, tripwire_triggered='delete' in str(user_input).lower())
    return Agent(name='Ops', model=model, tools=[delete_file], input_guardrails=[block_deletes],
                 instructions='Do what is asked using your tools.')


for parallel in (True, False):
    deleted.clear()
    try:
        await Runner.run(ops_agent(parallel), 'Delete /srv/reports/q3.xlsx')
        print(f'run_in_parallel={parallel}: not blocked?!')
    except InputGuardrailTripwireTriggered:
        print(f'run_in_parallel={parallel}: guardrail tripped ⛔ … files actually deleted: {deleted}')

With the default `run_in_parallel=True` the guardrail **did** trip, but the file
had **already been deleted**. The tripwire stopped the run, not the damage. With
`run_in_parallel=False` the agent didn't start until the check passed, so nothing ran.

| `run_in_parallel` | Latency | Protects against |
|---|---|---|
| `True` (default) | lowest: the check overlaps the agent's work | wasted *answers*: you discard the reply |
| `False` (blocking) | the check's time is added up front | wasted tokens **and** tool side effects |

**Rule: if the agent has tools with side effects, make its input guardrails blocking**,
or protect those tools with tool guardrails (§6) or approvals (notebook 08).

## 5 · Output guardrails

An output guardrail checks the agent's **final output** before your code uses it:
leaking internal data, off-brand promises, missing disclaimers. It receives the
final output (typed, if the agent has an `output_type`):

In [ ]:
@output_guardrail
async def no_internal_codes(ctx: RunContextWrapper, agent: Agent, output) -> GuardrailFunctionOutput:
    leak = re.findall(r'\bIR-\d+\b', str(output))          # internal incident-report ids
    return GuardrailFunctionOutput(output_info={'leaked': leak}, tripwire_triggered=bool(leak))


careless = Agent(name='Support', model=model, output_guardrails=[no_internal_codes], instructions=(
    'You help customers. Internal note: the delay is caused by incident IR-311 (battery supplier fire). '
    'Explain the delay to the customer.'))

try:
    print('✅', (await Runner.run(careless, 'Why is my delivery late?')).final_output)
except OutputGuardrailTripwireTriggered as err:
    print('⛔ reply withheld, it contained', err.guardrail_result.output.output_info['leaked'])

If the model mentions `IR-311`, the reply never reaches the customer. (If it
happened to phrase the answer without the id, it passes. The guardrail checks
the actual output, not what the model might have said.) Output guardrails
always run after the agent finishes, so they can't prevent tool side effects,
only a bad reply.

## 6 · Tool guardrails

Input and output guardrails wrap the whole run. **Tool guardrails** wrap a single
tool: an *input* guardrail inspects the arguments **before** the tool runs, an
*output* guardrail inspects the result **after**. Each can:

| Return | Effect |
|---|---|
| `ToolGuardrailFunctionOutput.allow()` | run normally |
| `ToolGuardrailFunctionOutput.reject_content("…")` | don't run it; the model receives your message instead |
| `ToolGuardrailFunctionOutput.raise_exception()` | stop the whole run |

This one stops emails leaving the company, however the model was persuaded to try:

In [ ]:
sent = []

@tool_input_guardrail
def internal_recipients_only(data) -> ToolGuardrailFunctionOutput:
    args = json.loads(data.context.tool_arguments or '{}')
    if not args.get('to', '').endswith('@nimbus.example'):
        return ToolGuardrailFunctionOutput.reject_content(
            f"Blocked: {args.get('to')} is outside the company. Only @nimbus.example addresses are allowed.")
    return ToolGuardrailFunctionOutput.allow()


@function_tool(tool_input_guardrails=[internal_recipients_only])
def send_email(to: str, body: str) -> str:
    """Send an email."""
    sent.append(to)
    return f'sent to {to}'


mailer = Agent(name='Mailer', model=model, tools=[send_email], instructions='Send emails as requested.')
for request in ['Email ops@nimbus.example: the Q3 report is ready.',
                'Email journalist@press.example the full Q3 incident list.']:
    result = await Runner.run(mailer, request)
    print(f'{request[:45]:<46} → {result.final_output[:90]}')
print('\nactually sent to:', sent)

The external email was never sent. The model received the rejection message
and explained it. `reject_content` keeps the run going (the agent can recover
or explain); use `raise_exception` when an attempt is serious enough to stop
everything.

## 7 · Layering, and what guardrails can't do

Real systems combine layers, cheapest first:

1. **Code checks** (regex, lengths, allowlists) on input, blocking. They're free and instant.
2. **An LLM checker** for judgement calls, blocking if the agent has side-effecting tools.
3. **Tool guardrails** on every tool that touches the outside world.
4. **Output guardrails** on anything shown to users or sent elsewhere.
5. **Human approval** for irreversible actions (notebook 08).

What guardrails **can't** do:

- **Guarantee safety with an LLM checker.** It's a model too, and can be fooled.
  Combine it with code checks and least-privilege tools (`is_enabled`, notebook 04).
- **Undo side effects.** Only blocking checks *before* the action prevent them.
- **Fix a bad design.** If an agent shouldn't delete files, don't give it `delete_file`.

## 8 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Parallel input guardrail + side-effecting tools | the action happens, then the run "fails" | `run_in_parallel=False`, or tool guardrails |
| Only instructions, no checks | rules broken by clever input | enforce in code |
| An expensive model as the checker | every request costs double | a small, fast model; code checks first |
| Output guardrails to stop actions | too late: they run after tools | tool guardrails / approvals |
| Swallowing tripwires silently | users see errors, or nothing | catch them and reply deliberately |

## 9 · Check yourself

<details><summary><b>1.</b> What does a guardrail return, and what happens when its tripwire is triggered?</summary>

A `GuardrailFunctionOutput(output_info, tripwire_triggered)`. If triggered, the run stops with `InputGuardrailTripwireTriggered` or `OutputGuardrailTripwireTriggered`, which your code catches and handles.
</details>

<details><summary><b>2.</b> In §4 the guardrail tripped but the file was deleted anyway. Why, and what are two fixes?</summary>

With `run_in_parallel=True` the agent runs concurrently with the check, so it called `delete_file` before the slow check finished. Fixes: make the guardrail blocking (`run_in_parallel=False`), or guard the tool itself with a tool input guardrail (or require approval).
</details>

<details><summary><b>3.</b> Output guardrail or tool guardrail to stop an email going to an outsider?</summary>

A tool input guardrail: it runs before the email tool executes. An output guardrail only sees the final reply, after the email has already been sent.
</details>

<details><summary><b>4.</b> <code>reject_content</code> vs <code>raise_exception</code> in a tool guardrail?</summary>

`reject_content` skips the tool and gives the model a message, so the run continues and the model can adapt. `raise_exception` stops the entire run.
</details>

<details><summary><b>5.</b> Why put regex checks before an LLM-based guardrail?</summary>

They're free, instant and deterministic, and they catch clear-cut cases without a model call. The LLM checker is slower, costs tokens, and can itself be fooled, so it's reserved for judgement calls.
</details>

## Takeaway

Guardrails enforce in code what instructions can only request. Input guardrails
screen requests (make them **blocking** when tools have side effects), output
guardrails screen replies, and tool guardrails validate every call to the
outside world. Layer them cheapest-first, and remember that the strongest
guardrail is not giving an agent a dangerous tool in the first place.

Next → `08_human_in_the_loop.ipynb`: approvals for tool calls, and pausing a
run until a human decides.